# TCAD-Driven Traveling-Wave Mach-Zehnder Modulator

This notebook reproduces the full electro-optic modulator workflow requested in
[gdsfactory/gsim#181](https://github.com/gdsfactory/gsim/issues/181) on
open-source solvers, following the classic CHARGE → MODE → circuit flow:

1. **Charge transport** (`gsim.tcad`, DEVSIM): Poisson + drift-diffusion on the
   waveguide cross-section gives carrier maps $n(x,y)$, $p(x,y)$ and the
   junction capacitance $C(V)$ per bias point.
2. **Validation**: the TCAD $C(V)$ is cross-checked against the retained
   analytic depletion model (`PNJunctionConfig`, Sze).
3. **Carrier→material coupling** (`gsim.common.carriers`): Soref/Nedeljkovic
   plasma dispersion gives $\Delta n(x,y)$, $\Delta\alpha(x,y)$ for optics and
   the Drude $\sigma(x,y)$ for RF.
4. **Carrier-aware optical modes** (`gsim.femwell`): the *same* mesh the charge
   solve used is loaded into femwell with a continuous carrier-perturbed
   $\varepsilon(x,y)$ — the graded-profile reference route. The **staircase
   route** bins the same carrier maps into piecewise-constant strips for the
   native Palace `BoundaryMode` solver.
5. **TW-MZM assembly** (`gsim.common.twmzm`): RF line parameters and the
   optical bias sweep combine into the traveling-wave transfer function —
   velocity mismatch, impedance matching, RF loss, EO bandwidth, and
   $V_\pi L$.

Install everything with the combined extra:

```bash
pip install 'gsim[modulator]'
```


## Phase-shifter cross-section

A lateral PN junction in a 220 nm silicon strip, split into four doped regions
along the junction axis: `n_pad | n_rib | p_rib | p_pad`. The ohmic contacts
sit on the outer pads, well away from the metallurgical junction, so the
contact boundary condition does not distort the junction electrostatics. One
gdsfactory component and one native-2D mesh feed **every** solver below.


In [ ]:
import gdsfactory as gf
import matplotlib.pyplot as plt
import meshio
import numpy as np

from gsim.common.cross_section import build_doped_cross_section
from gsim.common.stack.extractor import Layer
from gsim.common.stack.junction import PNJunctionConfig
from gsim.common.stack.materials import make_doped_materials

# --- Device parameters -------------------------------------------------------
NA_CM3 = 1e18  # acceptor doping, P side
ND_CM3 = 1e18  # donor doping, N side
RIB_HEIGHT = 0.22  # um
HALF_WIDTH = 0.3  # um, each junction flank
PAD_WIDTH = 0.3  # um, contact pads
CENTER_Y = -20.0  # um, junction position in the layout
LENGTH_UM = 10.0  # um, drawn length (the mode solve is 2D)

WAVELENGTH_UM = 1.55
N_SI = 3.476  # unperturbed silicon index at 1.55 um
N_GROUP_OPT = 3.8  # optical group index for the walk-off analysis
REVERSE_BIASES = [0.0, 0.5, 1.0, 1.5, 2.0]  # V on the cathode (= reverse bias)

gf.gpdk.PDK.activate()
comp = gf.Component()
wg = comp << gf.c.rectangle((LENGTH_UM, 0.4), centered=True, layer=(1, 0))
wg.y = CENTER_Y
slab = comp << gf.c.rectangle((LENGTH_UM, 100.0), centered=True, layer=(3, 0))
slab.y = -5.0

spans = {
    "n_pad": (CENTER_Y - HALF_WIDTH - PAD_WIDTH, CENTER_Y - HALF_WIDTH),
    "n_rib": (CENTER_Y - HALF_WIDTH, CENTER_Y),
    "p_rib": (CENTER_Y, CENTER_Y + HALF_WIDTH),
    "p_pad": (CENTER_Y + HALF_WIDTH, CENTER_Y + HALF_WIDTH + PAD_WIDTH),
}
layer_specs = {}
for i, (name, (y0, y1)) in enumerate(spans.items()):
    gds_layer = (30, i)
    rect = comp << gf.c.rectangle((LENGTH_UM, y1 - y0), layer=gds_layer)
    rect.y = (y0 + y1) / 2
    layer_specs[name] = Layer(
        name=name,
        gds_layer=gds_layer,
        zmin=0.0,
        zmax=RIB_HEIGHT,
        thickness=RIB_HEIGHT,
        material=name,
        layer_type="dielectric",
        mesh_resolution="fine",
    )
# Metal electrodes over the pads, drawn on an M1-style PDK layer: the
# charge solve binds its ohmic contacts to these real electrode layers
# (native-2D meshes conductors as boundary curves).
ELECTRODE_THICKNESS = 0.5
for j, (el_name, el_span) in enumerate(
    [("cathode_metal", spans["n_pad"]), ("anode_metal", spans["p_pad"])]
):
    gds_layer = (41, j)
    y0, y1 = el_span
    rect = comp << gf.c.rectangle((LENGTH_UM, y1 - y0), layer=gds_layer)
    rect.y = (y0 + y1) / 2
    layer_specs[el_name] = Layer(
        name=el_name,
        gds_layer=gds_layer,
        zmin=RIB_HEIGHT,
        zmax=RIB_HEIGHT + ELECTRODE_THICKNESS,
        thickness=ELECTRODE_THICKNESS,
        material="aluminum",
        layer_type="conductor",
        mesh_resolution="fine",
    )

materials = make_doped_materials([(n, 1.6e3) for n in spans], permittivity=11.9)
stack, _section = build_doped_cross_section(
    comp,
    axis="x",
    value=0.0,
    substrate_thickness=2.0,
    doping={"layer_specs": layer_specs, "materials": materials},
    verbose=False,
)
print("doped regions:", list(spans))

## Charge transport: Poisson + drift-diffusion (DEVSIM)

`ChargeTransportSim` follows the same sim-class idiom as the `palace`/`meep`
backends. Contacts and region-region interfaces are declared as layer pairs;
the native-2D mesh generator tags their shared curves as named line groups
that DEVSIM binds to. Doping is analytic (`StepDoping` here; Gaussian and
implant-like profiles are available).


In [ ]:
from gsim import tcad

sim = tcad.ChargeTransportSim()
sim.set_output_dir("./tcad-twmzm")
sim.set_stack(stack)
sim.set_geometry(comp)
sim.set_airbox(margin_x=2.0, margin_y=2.0, z_above=1.5, z_below=1.0, material="sio2")
# Charge transport only needs the doped slab between the contacts, so the
# meshed domain is clipped to a window around the junction.
sim.set_cross_section("x=0", window=(CENTER_Y - 1.5, CENTER_Y + 1.5))

# Ohmic contacts where the metal electrodes land on the pads; interfaces
# stitch the doped regions into one continuous device (the middle one is
# the PN junction).
sim.add_contact(name="anode", layer_a="p_pad", layer_b="anode_metal")
sim.add_contact(name="cathode", layer_a="n_pad", layer_b="cathode_metal")
sim.add_interface(name="junction", layer_a="p_rib", layer_b="n_rib")
sim.add_interface(name="p_link", layer_a="p_pad", layer_b="p_rib")
sim.add_interface(name="n_link", layer_a="n_pad", layer_b="n_rib")

for region in ("p_rib", "p_pad"):
    sim.add_doping(
        tcad.StepDoping(region=region, dopant_type="acceptor", concentration_cm3=NA_CM3)
    )
for region in ("n_rib", "n_pad"):
    sim.add_doping(
        tcad.StepDoping(region=region, dopant_type="donor", concentration_cm3=ND_CM3)
    )

sim.mesh(preset="coarse", refined_mesh_size=0.02, max_mesh_size=40.0, verbose=False)
print("shared mesh:", sim.mesh_path)

### Per-solver cross-section windows

One component, three meshed domains: the charge solve just clipped its
domain to the doped slab between the contacts (`window=` above), the
optical solve needs only a few-µm box around the rib, and the RF solve
keeps the full extent (the electrode staircase section below).
`set_cross_section` takes the in-plane and vertical intervals directly.

In [ ]:
from scipy.constants import speed_of_light as c0

from gsim.palace import BoundaryModeSim

optical_probe = BoundaryModeSim()
optical_probe.set_output_dir("./tcad-twmzm-optical-window")
optical_probe.set_stack(stack)
optical_probe.set_geometry(comp)
optical_probe.set_airbox(
    margin_x=2.0, margin_y=2.0, z_above=1.5, z_below=1.0, material="sio2"
)
optical_probe.set_cross_section(
    "x=0",
    window=(CENTER_Y - 2.0, CENTER_Y + 2.0),  # a few-um optical box
    window_z=(-1.0, 1.2),
)
optical_probe.set_boundary_mode(freq=c0 / (WAVELENGTH_UM * 1e-6), num_modes=1)
optical_probe.mesh(
    preset="coarse", refined_mesh_size=0.05, max_mesh_size=40.0, verbose=False
)


def _extent(mesh_path):
    pts = meshio.read(str(mesh_path)).points
    return (
        f"y in [{pts[:, 0].min():6.1f}, {pts[:, 0].max():6.1f}] um, "
        f"z in [{pts[:, 1].min():5.1f}, {pts[:, 1].max():5.1f}] um"
    )


print("charge  (doped-slab window):", _extent(sim.mesh_path))
print(
    "optical (rib-box window):   ", _extent(optical_probe._last_mesh_result.mesh_path)
)
print("RF keeps the full component extent - staircase section below.")

### Bias sweep

Positive cathode (n-side) bias reverse-biases the junction. Each bias point
returns the carrier maps on the mesh nodes, the terminal currents, and the
small-signal capacitance from a quasi-static AC solve.


In [ ]:
sweep = sim.sweep(REVERSE_BIASES, contact="cathode")
for point in sweep.points:
    print(
        f"V = {point.bias_v:4.1f} V:  C = {point.capacitance_f_per_m * 1e12:6.1f} pF/m,"
        f"  I_cathode = {point.currents_a_per_cm['cathode']:+.2e} A/cm"
    )

## TCAD C(V) versus the analytic depletion model

The retained `PNJunctionConfig` (Sze) path doubles as a validation reference:
on this abrupt symmetric junction the numeric small-signal $C(V)$ must track
$\varepsilon_s / W(V)$.


In [ ]:
junction = PNJunctionConfig(na_cm3=NA_CM3, nd_cm3=ND_CM3)
comparison = tcad.compare_capacitance(junction, sweep, height_um=RIB_HEIGHT)
print(f"max relative deviation: {comparison.max_relative_deviation:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    comparison.v_reverse, comparison.c_tcad_f_per_cm * 1e14, "o-", label="TCAD (DEVSIM)"
)
ax.plot(
    comparison.v_reverse,
    comparison.c_analytic_f_per_cm * 1e14,
    "s--",
    label="Analytic (Sze)",
)
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C (fF per um of length)")
ax.set_title("Junction capacitance: TCAD vs depletion approximation")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Carrier maps

The depletion region widens with reverse bias — the physics the single-strip
depletion model cannot resolve spatially.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for ax, point in zip(axes, (sweep.points[0], sweep.points[-1])):
    c = point.carriers
    tri = ax.tricontourf(
        c.x_um, c.y_um, np.log10(np.maximum(c.electrons_cm3, 1.0)), levels=30
    )
    ax.set_title(f"log10 n(x, y) at V = {point.bias_v} V")
    ax.set_xlabel("x (um)")
fig.colorbar(tri, ax=axes, label="log10 n (cm^-3)")
axes[0].set_ylabel("z (um)")
plt.show()

## Carrier → material coupling

Pure functions in `gsim.common.carriers` turn the carrier maps into what the
EM solvers consume: the Soref/Nedeljkovic index shift $\Delta n$ and
free-carrier absorption $\Delta\alpha$ for optics, and the Drude
$\sigma = q(\mu_n n + \mu_p p)$ for RF. The coefficients are overridable
for foundry calibration.


In [ ]:
from gsim.common.carriers import (
    PlasmaDispersionModel,
    carrier_absorption_cm,
    carrier_conductivity,
    carrier_index_shift,
)

model = PlasmaDispersionModel.nedeljkovic_1550()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for point in (sweep.points[0], sweep.points[-1]):
    c = point.carriers
    band = np.abs(c.y_um - RIB_HEIGHT / 2) < 0.03
    order = np.argsort(c.x_um[band])
    x = c.x_um[band][order]
    n = c.electrons_cm3[band][order]
    p = c.holes_cm3[band][order]
    label = f"V = {point.bias_v} V"
    axes[0].plot(x, carrier_index_shift(n, p, model=model), label=label)
    axes[1].plot(x, carrier_absorption_cm(n, p, model=model), label=label)
    axes[2].plot(x, carrier_conductivity(n, p), label=label)
axes[0].set_ylabel("$\\Delta n$")
axes[1].set_ylabel("$\\Delta\\alpha$ (1/cm)")
axes[2].set_ylabel("$\\sigma$ (S/m)")
for ax in axes:
    ax.set_xlabel("x (um)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Carrier-derived material response across the junction (mid-rib cut)")
plt.tight_layout()
plt.show()

## Carrier-aware optical modes (femwell, continuous $\varepsilon(x,y)$)

The femwell adapter loads the **identical** mesh the charge solve used and
projects a continuously varying carrier-perturbed permittivity onto a
per-element basis — the configuration Palace cannot express. Every element in
a silicon region gets $(n_{Si} + \Delta n(x,y))^2$ with the local absorption
as a negative imaginary part.


In [ ]:
from scipy.interpolate import LinearNDInterpolator

from gsim.common.carriers import permittivity_perturbation
from gsim.femwell import solve_modes

EPS_SIO2 = 1.444**2
SI_REGIONS = {"n_pad", "n_rib", "p_rib", "p_pad", "core", "slab90"}

mesh = meshio.read(str(sim.mesh_path))
tri_blocks = [
    (block.data, np.asarray(phys))
    for block, phys in zip(mesh.cells, mesh.cell_data["gmsh:physical"])
    if block.type == "triangle"
]
tris = np.vstack([d for d, _ in tri_blocks])
phys_tags = np.concatenate([p for _, p in tri_blocks])
centroids = mesh.points[tris][:, :, :2].mean(axis=1)
tag_to_name = {
    int(np.asarray(data)[0]): name
    for name, data in mesh.field_data.items()
    if int(np.asarray(data)[1]) == 2
}
element_names = np.array([tag_to_name[int(t)] for t in phys_tags])
silicon = np.isin(element_names, list(SI_REGIONS))


def carrier_epsilon(point):
    """Per-element eps: SiO2 background, carrier-perturbed silicon."""
    c = point.carriers
    dn_nodes = carrier_index_shift(c.electrons_cm3, c.holes_cm3, model=model)
    da_nodes = carrier_absorption_cm(c.electrons_cm3, c.holes_cm3, model=model)
    pts = np.column_stack([c.x_um, c.y_um])
    dn = LinearNDInterpolator(pts, dn_nodes, fill_value=0.0)(centroids)
    da = LinearNDInterpolator(pts, da_nodes, fill_value=0.0)(centroids)
    eps = np.full(len(centroids), EPS_SIO2, dtype=complex)
    for i in np.flatnonzero(silicon):
        eps[i] = permittivity_perturbation(
            n0=N_SI,
            dn=float(dn[i]),
            dalpha_cm=float(da[i]),
            wavelength_um=WAVELENGTH_UM,
        )
    return eps


n_eff_v = []
for point in sweep.points:
    modes = solve_modes(
        sim.mesh_path,
        epsilon=carrier_epsilon(point),
        wavelength_um=WAVELENGTH_UM,
        num_modes=1,
    )
    n_eff_v.append(complex(modes[0].n_eff))
    print(f"V = {point.bias_v:4.1f} V:  n_eff = {n_eff_v[-1]:.6f}")

voltages = np.array(REVERSE_BIASES)
dn_eff = np.array([n.real - n_eff_v[0].real for n in n_eff_v])
# exp(+i omega t): Im(n_eff) < 0 is loss; alpha[dB/cm] = 40 pi Im / (ln10 lambda)
alpha_opt_db_cm = np.array(
    [40 * np.pi * abs(n.imag) / (np.log(10) * WAVELENGTH_UM * 1e-4) for n in n_eff_v]
)

### Phase-shift efficiency $V_\pi L$ and bias-dependent optical loss

The mode/carrier overlap is fully resolved, so the efficiency reflects the
true depletion-edge movement rather than a uniform-strip estimate.


In [ ]:
from gsim.common.twmzm import vpi_length_vcm

vpi_l = vpi_length_vcm(voltages, dn_eff, wavelength_um=WAVELENGTH_UM)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(voltages, dn_eff * 1e5, "o-")
axes[0].set_ylabel("$\\Delta n_{eff} \\times 10^{-5}$")
axes[1].plot(voltages, vpi_l, "o-")
axes[1].set_ylabel("$V_\\pi L$ (V cm)")
axes[2].plot(voltages, alpha_opt_db_cm, "o-")
axes[2].set_ylabel("optical loss (dB/cm)")
for ax in axes:
    ax.set_xlabel("Reverse bias (V)")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"V_pi L at {voltages[-2]:.1f} V: {vpi_l[-2]:.2f} V cm")

## Staircase route: carrier strips for Palace BoundaryMode

Palace accepts only piecewise-constant materials per domain, so the same
carrier maps are binned into $N$ strips
(`strip_averages_from_nodes` → `make_staircase_profile`). Each strip becomes a
patterned-dielectric Palace domain; $N=1$ recovers the uniform-strip model and
increasing $N$ converges to the continuous profile solved above. Here we build
the $N=5$ staircase mesh plus flanking metal electrodes and write the
Palace config; the next section solves the same mesh with femwell. The
electrodes enter as finite-conductivity Drude *volumes* (not the usual PEC
curves) so the mode solver can integrate the electrode current for $Z_0$.


In [ ]:
from gsim.common.stack.staircase import (
    make_staircase_profile,
    strip_averages_from_nodes,
)
from gsim.palace import BoundaryModeSim

point = sweep.points[-1]  # staircase the highest-bias carrier maps
c = point.carriers
N_STRIPS = 5
edges, n_means = strip_averages_from_nodes(
    c.x_um,
    c.electrons_cm3,
    n_strips=N_STRIPS,
    h_min=CENTER_Y - HALF_WIDTH,
    h_max=CENTER_Y + HALF_WIDTH,
    v_um=c.y_um,
    v_range=(0.0, RIB_HEIGHT),
)
_edges, p_means = strip_averages_from_nodes(
    c.x_um,
    c.holes_cm3,
    n_strips=N_STRIPS,
    h_min=CENTER_Y - HALF_WIDTH,
    h_max=CENTER_Y + HALF_WIDTH,
    v_um=c.y_um,
    v_range=(0.0, RIB_HEIGHT),
)

stair_comp = gf.Component()
wg2 = stair_comp << gf.c.rectangle((LENGTH_UM, 0.4), centered=True, layer=(1, 0))
wg2.y = CENTER_Y
staircase = make_staircase_profile(
    stair_comp,
    length=LENGTH_UM,
    edges=edges,
    n_strips_cm3=n_means,
    p_strips_cm3=p_means,
    base_layer=(40, 0),
    zmin=0.0,
    zmax=RIB_HEIGHT,
    target="rf",
)
# CPS electrodes flanking the junction, as high-sigma volumetric regions.
ELECTRODE_WIDTH = 2.0
SIGMA_AL = 3.8e7
electrode_spans = {
    "electrode_n": (CENTER_Y - HALF_WIDTH - ELECTRODE_WIDTH, CENTER_Y - HALF_WIDTH),
    "electrode_p": (CENTER_Y + HALF_WIDTH, CENTER_Y + HALF_WIDTH + ELECTRODE_WIDTH),
}
for j, (el_name, (y0, y1)) in enumerate(electrode_spans.items()):
    gds_layer = (42, j)
    rect = stair_comp << gf.c.rectangle((LENGTH_UM, y1 - y0), layer=gds_layer)
    rect.y = (y0 + y1) / 2
    staircase["layer_specs"][el_name] = Layer(
        name=el_name,
        gds_layer=gds_layer,
        zmin=0.0,
        zmax=ELECTRODE_THICKNESS,
        thickness=ELECTRODE_THICKNESS,
        material=el_name,
        layer_type="dielectric",
        mesh_resolution="fine",
    )
staircase["materials"].update(
    make_doped_materials(
        [(n, SIGMA_AL) for n in electrode_spans],
        permittivity=1.0,
        source_prefix="electrode",
    )
)

stair_stack, _ = build_doped_cross_section(
    stair_comp,
    axis="x",
    value=0.0,
    substrate_thickness=2.0,
    doping=staircase,
    verbose=False,
)

bsim = BoundaryModeSim()
bsim.set_output_dir("./tcad-twmzm-staircase")
bsim.set_stack(stair_stack)
bsim.set_geometry(stair_comp)
bsim.set_airbox(margin_x=2.0, margin_y=2.0, z_above=1.5, z_below=1.0, material="sio2")
bsim.set_cross_section("x=0")
bsim.set_boundary_mode(freq=50e9, num_modes=1)
bsim.mesh(preset="coarse", refined_mesh_size=0.05, max_mesh_size=40.0, verbose=False)
bsim.write_config()

volumes = bsim._last_mesh_result.groups["volumes"]
strip_domains = sorted(v for v in volumes if v.startswith("strip_"))
print("Palace strip domains:", strip_domains)
print("per-strip sigma (S/m):", np.round(staircase["strips"]["sigma_s_per_m"], 1))

## RF mode with carrier-derived $\sigma(x,y)$: solver route

femwell solves the electrode-loaded staircase cross-section directly: each
strip carries the Drude conductivity binned from the carrier maps and the
electrodes their metal conductivity. The propagating mode with the largest
effective index is the junction-loaded slow-wave line mode; its complex
$n_{eff}$ gives $\gamma = \alpha + i\beta$ per frequency, and $Z_0$
comes from the Marks-Williams power-current integral
$Z_0 = 2P / |I|^2$ with $I$ integrated over one electrode — no analytic
line model anywhere in this chain.

In [ ]:
from gsim.femwell.adapter import epsilon_by_region, z0_power_current

RF_FREQS = np.array([10e9, 20e9, 40e9, 60e9, 80e9, 100e9])
rf_mesh_path = bsim._last_mesh_result.mesh_path

rf_mesh = meshio.read(str(rf_mesh_path))
rf_phys = np.concatenate(
    [
        np.asarray(phys)
        for block, phys in zip(
            rf_mesh.cells, rf_mesh.cell_data["gmsh:physical"], strict=False
        )
        if block.type == "triangle"
    ]
)
rf_tags = {
    name: int(np.asarray(data)[0])
    for name, data in rf_mesh.field_data.items()
    if int(np.asarray(data)[1]) == 2
}
signal_elements = np.flatnonzero(rf_phys == rf_tags["electrode_p"])

n_eff_rf, z0_rf = [], []
for f_hz in RF_FREQS:
    eps_f = epsilon_by_region(rf_mesh_path, bsim.stack, frequency_hz=f_hz)
    rf_modes = solve_modes(
        rf_mesh_path,
        epsilon=eps_f,
        wavelength_um=c0 / f_hz * 1e6,
        num_modes=4,
        metallic_boundaries=True,
        n_guess=3.0,
    )
    candidates = [
        m
        for m in rf_modes
        if complex(m.n_eff).real > 1.0
        and abs(complex(m.n_eff).imag) < complex(m.n_eff).real
    ]
    mode = max(candidates, key=lambda m: complex(m.n_eff).real)
    n_eff_rf.append(complex(mode.n_eff))
    z0_rf.append(
        z0_power_current(mode, frequency_hz=f_hz, current_elements=signal_elements)
    )
    alpha_db_cm = 8.686 * abs(n_eff_rf[-1].imag) * 2 * np.pi * f_hz / c0 / 100
    print(
        f"f = {f_hz / 1e9:5.0f} GHz:  n_RF = {n_eff_rf[-1].real:5.2f},  "
        f"alpha_RF = {alpha_db_cm:5.1f} dB/cm,  Z0 = {z0_rf[-1].real:5.1f} ohm"
    )

n_eff_rf = np.array(n_eff_rf)
z0_rf = np.array(z0_rf)

## Analytic cross-check: loaded-line model with the TCAD C(V)

The traveling-wave electrode is a periodically loaded transmission line: the
unloaded CPW contributes $L$, $C_{line}$, and the phase-shifter junction loads
it with the bias-dependent $C_j(V)$ from the charge solve. This is the
standard velocity-engineering picture — the loading slows the RF wave, and the
designer tunes the fill factor so $n_{RF}$ matches the optical group index.

This classic velocity-engineering model is kept as the analytic
cross-check of the solver route above (and it is what gives the designer
$n_{RF}$ *versus bias*, since the mode solve above ran at the staircased
operating bias).

In [ ]:
from scipy.constants import speed_of_light as c0

# Unloaded electrode (typical thin-film CPW values)
Z0_UNLOADED = 90.0  # ohm
N_UNLOADED = 2.2  # unloaded RF index
L_PUL = Z0_UNLOADED * N_UNLOADED / c0  # H/m
C_LINE = N_UNLOADED / (Z0_UNLOADED * c0)  # F/m
FILL = 0.8  # fraction of the line loaded by the junction
ALPHA0_DB_CM = 0.6  # RF loss at 1 GHz (dB/cm), ~sqrt(f) skin-effect scaling

freq = np.linspace(1e9, 100e9, 200)


def loaded_line(c_j_f_per_m):
    c_tot = C_LINE + FILL * c_j_f_per_m
    n_rf = c0 * np.sqrt(L_PUL * c_tot)
    z0 = np.sqrt(L_PUL / c_tot)
    alpha_np_m = (
        ALPHA0_DB_CM * np.sqrt(freq / 1e9) * 100.0 / 8.686  # dB/cm -> Np/m
    )
    return n_rf, z0, alpha_np_m


c_j = sweep.capacitance_f_per_m
for v, cj in zip(voltages, c_j):
    n_rf, z0, _ = loaded_line(cj)
    print(
        f"V = {v:4.1f} V:  C_j = {cj * 1e12:6.1f} pF/m  ->  n_RF = {n_rf:.2f}, Z0 = {z0:.1f} ohm"
    )

## Velocity-mismatch analysis

The explicit "sample of velocity mismatch" from #181: the loaded RF index
versus the optical group index, and the walk-off-limited bandwidth
$f_{3dB} = 1.39\,c / (\pi L |n_{RF} - n_g|)$ as a function of electrode
length.


In [ ]:
from gsim.common.twmzm import walkoff_bandwidth

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
n_rf_v = np.array([loaded_line(cj)[0] for cj in c_j])
axes[0].plot(voltages, n_rf_v, "o-", label="$n_{RF}$ (loaded line)")
axes[0].axhline(N_GROUP_OPT, color="k", ls="--", label="$n_g$ optical")
axes[0].axhline(
    np.mean(n_eff_rf.real),
    color="C2",
    ls=":",
    label=f"$n_{{RF}}$ femwell staircase (V = {REVERSE_BIASES[-1]:.0f} V)",
)
axes[0].set_xlabel("Reverse bias (V)")
axes[0].set_ylabel("index")
axes[0].set_title("Velocity matching vs bias")
axes[0].legend()
axes[0].grid(alpha=0.3)

lengths_mm = np.linspace(1, 10, 40)
bias_idx = 2  # operating point
mismatch = abs(n_rf_v[bias_idx] - N_GROUP_OPT)
if mismatch > 0:
    f3db = [
        walkoff_bandwidth(
            length_m=length * 1e-3, n_rf=n_rf_v[bias_idx], n_opt=N_GROUP_OPT
        )
        / 1e9
        for length in lengths_mm
    ]
    axes[1].plot(lengths_mm, f3db)
axes[1].set_xlabel("electrode length (mm)")
axes[1].set_ylabel("walk-off $f_{3dB}$ (GHz)")
axes[1].set_title(
    f"Walk-off limit at V = {voltages[bias_idx]} V "
    f"($|n_{{RF}} - n_g|$ = {mismatch:.2f})"
)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Full device report: EO bandwidth and figures of merit

`twmzm_figures_of_merit` combines the RF line parameters with the optical
bias sweep into the standard single-drive traveling-wave response, including
RF loss and source/load reflections. The RF side is now entirely
solver-derived: $\gamma$ from the femwell staircase mode's complex
$n_{eff}$ and $Z_0$ from the power-current integral, interpolated onto the
dense frequency grid.


In [ ]:
from gsim.common.twmzm_report import (
    OpticalPhaseSweep,
    RFLineParams,
    line_params_from_neff,
    twmzm_figures_of_merit,
)

LENGTH_M = 3e-3  # 3 mm electrode

# Solver-derived line parameters (gamma from n_eff, Z_0 from the
# power-current integral), interpolated onto the dense frequency grid.
rf_solved = line_params_from_neff(RF_FREQS, n_eff_rf, z0_ohm=z0_rf)
rf = RFLineParams(
    freq_hz=freq,
    n_rf=np.interp(freq, RF_FREQS, rf_solved.n_rf),
    alpha_rf_np_m=np.interp(freq, RF_FREQS, rf_solved.alpha_rf_np_m),
    z0_ohm=np.interp(freq, RF_FREQS, rf_solved.z0_ohm.real)
    + 1j * np.interp(freq, RF_FREQS, rf_solved.z0_ohm.imag),
)
z0_op = float(np.mean(rf.z0_ohm.real))  # terminate in the line impedance
optical = OpticalPhaseSweep(
    voltages_v=voltages,
    dn_eff=dn_eff,
    alpha_opt_db_cm=alpha_opt_db_cm,
    wavelength_um=WAVELENGTH_UM,
    n_group=N_GROUP_OPT,
)
report = twmzm_figures_of_merit(
    rf, optical, length_m=LENGTH_M, z_load_ohm=z0_op, z_gen_ohm=50.0
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(report.freq_hz / 1e9, 20 * np.log10(np.abs(report.response)))
ax.axhline(20 * np.log10(1 / np.sqrt(2)), color="k", ls=":", label="-3 dB (EO)")
if report.bandwidth_3db_hz:
    ax.axvline(report.bandwidth_3db_hz / 1e9, color="r", ls="--")
ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("EO response (dB)")
ax.set_title(f"TW-MZM response, L = {LENGTH_M * 1e3:.0f} mm, matched load")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("=== TW-MZM figures of merit ===")
if report.bandwidth_3db_hz:
    print(f"EO 3 dB bandwidth:        {report.bandwidth_3db_hz / 1e9:.1f} GHz")
if report.walkoff_bandwidth_hz:
    print(f"walk-off limit:           {report.walkoff_bandwidth_hz / 1e9:.1f} GHz")
print(f"velocity mismatch:        n_RF - n_g = {report.velocity_mismatch[0]:+.2f}")
print(f"characteristic impedance: {report.z0_ohm[0].real:.1f} ohm")
print(f"V_pi L (operating bias):  {report.vpi_l_vcm[bias_idx]:.2f} V cm")
rlgc = report.rlgc
print(
    f"RLGC at 10 GHz: R = {np.interp(10e9, freq, rlgc['R']):.0f} ohm/m, "
    f"L = {np.interp(10e9, freq, rlgc['L']) * 1e9:.0f} nH/m, "
    f"C = {np.interp(10e9, freq, rlgc['C']) * 1e12:.0f} pF/m"
)

## Summary

From one gdsfactory component and one shared cross-section mesh:

- **DEVSIM** solved the junction charge transport per bias point — carrier
  maps and $C(V)$ validated against the analytic Sze model;
- **femwell** solved the optical mode with the continuous carrier-perturbed
  index — $\Delta n_{eff}(V)$, optical loss, and $V_\pi L$;
- the **staircase route** turned the same carrier maps into piecewise-constant
  Palace `BoundaryMode` domains, and **femwell** solved the electrode-loaded
  RF mode of that staircase — $\gamma$ from the solver, $Z_0$ from the
  Marks-Williams power-current integral;
- the **TW-MZM assembly** produced the velocity-mismatch analysis, RLGC line
  parameters, and the electro-optic bandwidth from the solver-derived line
  parameters, with the loaded-line model kept as the analytic cross-check;
- the charge solve used **real metal electrode layers** for its contacts and
  a clipped **cross-section window** (with the optical-box window shown
  alongside).

This is the open-source counterpart of the commercial CHARGE → MODE → circuit
modulator workflow requested in gdsfactory/gsim#181.
